# Focus Engine - Exploratory Data Analysis (EDA)

This notebook performs EDA on the sliding window dataset (`dataset.csv`) comprising real focus sessions and bootstrapped synthetic sessions. It validates the synthetic archetypes, reviews feature correlations, checks for outliers, and maps local hour distributions.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Setup paths
script_dir = os.getcwd()
dataset_path = os.path.join(script_dir, "data", "dataset.csv")
output_dir = os.path.join(script_dir, "eda_output")
os.makedirs(output_dir, exist_ok=True)

print(f"Reading dataset from: {dataset_path}")

In [ ]:
# 1. Load Dataset
df = pd.read_csv(dataset_path)
print(f"Total Rows in sliding training window: {len(df)}")
df.head()

In [ ]:
# 2. Summary Statistics
df.describe().T

### 3. Null Handling Strategy
1. **Drop Target Nulls**: Rows missing `focus_score` are dropped entirely, as we cannot train supervised models without a target label.
2. **Impute Feature Nulls**: Impute numeric features with their **Median** values. Median is preferred over Mean because it is robust to outliers, ensuring that metric peaks (e.g. huge KPM spikes or unusually long durations) do not skew the baseline imputations.

In [ ]:
# Apply target drop
initial_len = len(df)
df = df.dropna(subset=['focus_score'])
print(f"Dropped {initial_len - len(df)} rows missing focus_score.")

# Apply feature median imputation
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col in ['focus_score', 'is_synthetic', 'timestamp']:
        continue
    null_mask = df[col].isnull()
    if null_mask.any():
        median_val = df[col].median()
        df.loc[null_mask, col] = median_val
        print(f"Imputed {null_mask.sum()} nulls in '{col}' with median: {median_val}")

### 4. Outlier Audits
Check for impossible durations (> 12 hours or <= 0s) and impossible average typing speeds (> 400 KPM).

In [ ]:
duration_outliers = df[(df['session_duration'] <= 0) | (df['session_duration'] > 43200)]
print(f"Anomalous durations (<=0s or >12hr): {len(duration_outliers)}")

kpm_outliers = df[df['avg_kpm'] > 400]
print(f"Anomalous KPM values (>400 KPM): {len(kpm_outliers)}")

### 5. Overlap Histogram (Focus Score Distributions)
Splits the histogram color-wise: Teal for real sessions (if any exist) and Coral for synthetic sessions.

In [ ]:
real_df = df[df['is_synthetic'] == 0]
syn_df = df[df['is_synthetic'] == 1]

plt.figure(figsize=(10, 6))
if len(real_df) > 0:
    sns.histplot(data=real_df, x="focus_score", color="teal", label=f"Real Sessions ({len(real_df)})", kde=True, alpha=0.6, bins=15)
sns.histplot(data=syn_df, x="focus_score", color="coral", label=f"Synthetic Sessions ({len(syn_df)})", kde=True, alpha=0.6, bins=15)

plt.title("Focus Score Distribution (Real vs Synthetic Overlap)", fontsize=14, pad=15)
plt.xlabel("Focus Score", fontsize=12)
plt.ylabel("Frequency Count", fontsize=12)
plt.xlim(0, 100)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

### 6. Correlation Matrix Heatmap
Validates features, ensuring that `avg_buffer`, `focus_time`, and other metrics correctly correlate with the target `focus_score`.

In [ ]:
features = [
    "avg_buffer", "min_buffer", "max_buffer", "focus_time", 
    "attention_time", "avg_kpm", "mouse_activity", "pause_count", 
    "app_switches", "session_duration", "hour_of_day", "day_of_week", 
    "session_mode_is_standard", "focus_score"
]

plt.figure(figsize=(12, 10))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, cmap="coolwarm", annot=True, fmt=".2f", square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.title("Feature Correlation Matrix (Sliding Window)", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

### 7. Time of Day vs Focus Score (Real Only)
Scatter plot mapping hour of day vs focus score using ONLY real sessions, since synthetic timestamps are artificially backdated.

In [ ]:
plt.figure(figsize=(10, 6))
if len(real_df) > 0:
    sns.scatterplot(data=real_df, x="hour_of_day", y="focus_score", color="teal", s=100, alpha=0.8, edgecolor="k")
    plt.title("Hour of Day vs Focus Score (Real Sessions Only)", fontsize=14, pad=15)
    plt.xlim(-0.5, 23.5)
    plt.xticks(range(24))
else:
    plt.text(0.5, 0.5, "No Real Sessions Logged Yet\n(Awaiting user focus sessions)", 
             horizontalalignment='center', verticalalignment='center', fontsize=14, color='gray')
    plt.title("Hour of Day vs Focus Score (Real Sessions - Empty State)", fontsize=14, pad=15)
plt.xlabel("Hour of Day (Local Time)", fontsize=12)
plt.ylabel("Focus Score", fontsize=12)
plt.ylim(-5, 105)
plt.tight_layout()
plt.show()